# 🎯 [K12_M02] 행정 절차 FAQ 정리하기
### 학생 실습 워크북 & 프롬프트 설계 가이드 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K12/k12_m02_student.ipynb)

---

## 📌 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 / 과제** | 일반행정·부대관리 (Level 1) / K12 행정 업무 FAQ 챗봇 설계 |
| **미션** | 미션 02. 행정 절차 FAQ 정리하기 (소요시간: 약 1.5시간) |
| **실습 유형** | 생성형 AI 시스템 프롬프트 엔지니어링 및 팩트체크 검증 워크북 |

> **⚠️ [교육용 가상 자료 원칙]**
> 본 실습의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> 실제 군사 기밀이나 부대 세부 인적·작전 정보는 외부 AI 도구에 절대 입력하지 않습니다.



In [ ]:
# ============================================================================
# [환경 설정 및 헬퍼 함수] 마크다운 시각화 렌더러 및 가상 데이터셋 로더
# ============================================================================
from IPython.display import display, Markdown
import json
import os

def load_dataset(mission_num):
    """미션 번호(1~10)에 해당하는 가상 데이터셋 JSON을 로드하여 반환합니다."""
    filename = f"k12_m{mission_num:02d}_dataset.json"
    for base in [".", "uploads/K12", "../uploads/K12"]:
        filepath = os.path.join(base, filename)
        if os.path.exists(filepath):
            with open(filepath, "r", encoding="utf-8") as f:
                return json.load(f)
    print(f"⚠️ 데이터셋 파일을 찾을 수 없습니다: {filename}")
    return None

def show_dataset_summary(mission_num):
    """해당 미션의 데이터셋 주요 항목을 요약 출력합니다."""
    data = load_dataset(mission_num)
    if data:
        display(Markdown(f"**[데이터셋 로드 성공]** `{data.get('title', '')}` ({data.get('mission', '')})"))
        keys = [k for k in data.keys() if k not in ["project", "mission", "title"]]
        print(f"포함 항목: {', '.join(keys)}")

def show_result(mission_title, prompt, result):
    """학습자가 입력한 프롬프트와 AI 답변 및 검증 결과를 시각적으로 렌더링하는 헬퍼 함수"""
    p_text = prompt.strip() if prompt.strip() else "(작성된 프롬프트가 없습니다)"
    r_text = result.strip() if result.strip() else "(기록된 결과가 없습니다)"
    
    md_content = f"""
---
### 📌 [{mission_title}] 실습 기록 리포트

#### 💬 입력 프롬프트
```text
{p_text}
```

#### 📋 AI 생성 결과 및 팩트체크 검증 내역
{r_text}
---
"""
    display(Markdown(md_content))

print("✅ 워크북 시각화 헬퍼 함수 및 가상 데이터셋 로더(k12_m01~m10) 준비 완료!")



---

## 🎯 미션 02. 행정 절차 FAQ 정리하기
**소요시간**: 약 1.5시간

![미션 2 비정형 행정 규정의 7대 표준 컬럼 FAQ 테이블 구조화 및 기한·서류 무결성 검증 다이어그램](/uploads/common/K12_m02_faq_data_structuring.png)

### 📋 미션 개요 및 배경
생성형 AI가 허위 정보 없이 정확한 행정 절차를 안내할 수 있도록 비정형 질의응답을 정형화된 참조 데이터로 체계화합니다. 가상 한빛대대 행정 FAQ 20건을 번호·분류·질문·단계별 절차·필요 서류·담당 부서·연락처 항목의 표준 테이블로 구조화하고, 접수 기한이나 추첨 조건 등 주요 제약 조건을 명확히 표시합니다. AI의 검색 증강(RAG) 기반이 되는 비정형 텍스트의 데이터 정제 및 메타데이터 구조화 역량을 체득합니다.

### 🎯 미션 학습목표
가상 FAQ 20개를 번호·분류·질문·절차(단계)·필요 서류·담당 부서·연락처 항목으로 정리합니다. 기한이나 조건이 있는 항목(접수 기한, 20% 초과 시 추첨 등)은 따로 표시합니다. **이를 통해 비정형 행정 안내문을 AI가 정확히 검색하고 파싱할 수 있는 표준 데이터 테이블로 구조화하는 데이터 정제 및 메타데이터 설계 역량을 익힙니다.**

### 📑 실습 가이드 및 세부 단계 (프롬프트 & 검증 포인트)
#### 🔹 Step 1. FAQ 20개를 표 항목에 맞춰 확인한다.
#### 이 단계의 역할

가상 FAQ 20건을 AI가 검색·참조하기 적합한 표준 표(번호, 분류, 질문, 단계별 절차, 필요 서류, 담당 부서, 연락처)로 정리합니다.

#### 해야 할 일

- 가상 한빛대대 행정 FAQ 20건 원문을 꼼꼼히 확인합니다.

- [번호, 분류, 질문, 절차(단계별), 필요 서류, 담당 부서, 연락처] 7개 표준 열로 표 서식을 구성합니다.

- AI에게 비정형 줄글 FAQ를 표준 마크다운 표로 변환하도록 요청하고 필드를 원문과 대조합니다.

#### 💡 AI 프롬프트 질의 예시

> **[비정형 행정 FAQ 표 변환 질의]:** "다음 가상 한빛대대 행정 FAQ 20건의 줄글 안내문을 [번호, 분류, 질문, 절차(단계별), 필요 서류, 담당 부서, 연락처] 7개 열로 구성된 마크다운 표로 변환해 줘. 절차는 '1단계 → 2단계 → 3단계' 형태로 명확히 구분해 줘.
[FAQ 원문]: {가상 FAQ 20건 원문}"

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **전수 변환 확인:** F01부터 F20까지 20개 항목이 빠짐없이 표에 수록되었는가?
- **단계별 절차 분할:** 절차가 단순 텍스트 나열이 아닌 단계별(1·2·3단계) 순서로 명확히 나뉘었는가?

#### 핵심 확인 지표

> 표준 7개 컬럼 행정 업무 FAQ 완성표

#### 학습자가 제출해야 하는 내용

- 행정 업무 FAQ 표준 표 완성본

#### 🔹 Step 2. 기한·조건이 있는 항목을 표시한다.
#### 이 단계의 역할

휴가 신청 접수 기한(출타 3일 전 등), 정원 초과 시 추첨 조건, 신청 자격 등 조건부 규칙을 명시적으로 식별하여 표기합니다.

#### 해야 할 일

- FAQ 20건의 원문에서 'D-3일까지', '초과 시 추첨', '분기별 1회' 등 기한·조건 문구를 색출합니다.

- 표의 절차 및 주의사항 열에 기한과 자격 조건을 굵은 글씨나 별도 표기로 강조합니다.

- AI에게 조건부 예외 규칙이 존재하는 항목 목록을 추출하도록 질의하여 검증합니다.

#### 💡 AI 프롬프트 질의 예시

> **[행정 조건 및 기한 추출 질의]:** "정리된 행정 FAQ 20건 중 신청 기한(D-Day), 사전 승인 요건, 인원 정원/추첨 조건 등 '조건부 제약'이 포함된 항목을 모두 찾아내고, 해당 제약 사항을 명확히 요약해 줘."

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **기한·조건 누락 점검:** F02(정기휴가 접수 기한), F03(포상휴가 승인 시기), F08(자기개발 지원금 신청 기간) 등 주요 제약이 누락 없이 표기되었는가?
- **조건 표현의 정확성:** 날짜, 시간, 인원 비율(예: 20% 초과 시) 숫자가 원문과 정확히 일치하는가?

#### 핵심 확인 지표

> 기한·조건 명시 표기 및 조건 분석표

#### 학습자가 제출해야 하는 내용

- 기한·조건이 강조 표기된 FAQ 표

#### 🔹 Step 3. 담당 부서와 연락처가 빠진 항목이 없는지 본다.
#### 이 단계의 역할

질문에 대한 최종 확인 및 예외 문의를 처리할 수 있도록 20개 항목 전수의 담당 부서 및 가상 내선 번호를 철저히 검증합니다.

#### 해야 할 일

- FAQ 20개 항목 전수의 '담당 부서'와 '연락처' 컬럼을 전수 검사합니다.

- 결측값(공란, '미정', '행정반' 등 모호한 표기)이 있는 항목을 찾아 구체화합니다.

- AI에게 부서-업무 매칭이 부대 조직 구조에 부합하는지 교차 점검을 요청합니다.

#### 💡 AI 프롬프트 질의 예시

> **[담당 부서 및 내선 번호 결측 점검 질의]:** "행정 FAQ 20건 중 '담당 부서'나 '연락처(내선)'가 누락되거나 '해당 없음'으로 모호하게 적힌 항목이 있는지 검증하고, 결측 항목에 대해 적절한 부서(행정과, 인사과, 군수과, 통신소대 등)를 매칭해 줘."

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **완전성(Completeness):** 20개 항목 전부에 담당 부서명과 가상 내선 번호(예: 내선 1234)가 채워졌는가?
- **업무 적합성:** 증명서는 인사과, 시설은 군수과/영외수리반, 계정은 통신소대 등 업무 소관이 합리적인가?

#### 핵심 확인 지표

> 부서 및 연락처 결측치 0건 검증 완료표

#### 학습자가 제출해야 하는 내용

- 부서/연락처 검증 완료 FAQ 표

#### 🔹 Step 4. 심화 도전 과제 및 최적화
#### 이 단계의 역할

행정 절차상 첨부 서식(서식 번호, 양식 파일명) 열을 추가하여 데이터 구조를 한 단계 더 정밀화합니다.

#### 해야 할 일

- 주요 신청서(휴가신청서, 자격증지원신청서 등)의 가상 서식 번호를 매칭합니다.

- AI에게 표에 '제출 서식 파일명' 컬럼을 추가했을 때의 이점을 질의합니다.

#### 💡 AI 프롬프트 질의 예시

> **[FAQ 서식 메타데이터 확장 질의]:** "행정 FAQ 표에 '필요 서식 번호' 및 '다운로드 경로/양식명' 컬럼을 추가하려 해. AI 챗봇이 서식 작성법까지 연계 안내할 수 있도록 표 구조를 고도화하는 방안을 제안해 줘."

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **보안 준수:** 실제 군 내부 양식이 아닌 교육용 가상 서식 명칭만을 부여하였는가?

#### 핵심 확인 지표

> 서식 메타데이터가 보강된 심화 FAQ 표

#### 학습자가 제출해야 하는 내용

- 심화 도전 과제 메모

### 📦 미션 필수 학습자료 및 팁
**미션 수행 필수 학습자료**

- **가상 한빛대대 행정 절차 안내 원문 20건 (MD):** 가상 한빛대대 행정 규정, 절차, 필요 서류 줄글 안내 데이터
- **표준 행정 업무 FAQ 마크다운 테이블 서식:** 번호, 분류, 질문, 단계별 절차, 필요 서류, 담당 부서, 연락처 7대 표준 열 템플릿
- **[학습 가이드] 비정형 행정 지침의 표(Table) 구조화 및 메타데이터 정제 기법:** 생성형 AI가 절차 순서와 조건을 오독하지 않도록 단계를 분할하고 제약 사항을 명시하는 데이터 전처리 가이드

**실습 팁:** '출타 3일 전 제출(기한)'이나 '정원 20% 초과 시 추첨(조건)' 같은 제약 조건은 AI가 누락하기 쉬우므로 별도 열이나 굵은 글씨로 반드시 강조 표기하세요.

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 정보 구조화 — AI 검색에 최적화된 7대 표준 컬럼으로 FAQ를 정리하고 기한·조건 및 부서 메타데이터를 누락 없이 표시했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 20건 표준 행정 업무 FAQ 완성표 완성도 충족

- 단계별 절차(1·2·3단계) 및 필요 서류 누락 0건

- 주요 기한·조건부 예외 규칙 100% 명시

- 20개 전 항목 담당 부서 및 가상 내선 번호 결측치 0건

#### 채점자 판정 메모

> F01(출타 5일 전)과 F02(수여 6개월 내, D-3)의 기한 숫자가 원문과 정확히 일치하는지 집중 점검한다.



In [ ]:
# [미션 02.] 학생 실습 - 프롬프트 및 결과 기록 셀
# 💡 [데이터셋 확인 팁]: 아래 주석을 해제하여 본 미션의 가상 데이터셋 요약을 확인할 수 있습니다.
# show_dataset_summary(2)

# TODO 1: 위 실습 가이드와 AI 질의 예시를 참고하여 작성한 프롬프트를 입력하세요.
mission_02_prompt = """
# 여기에 본인이 설계한 프롬프트를 작성하세요.
"""

# TODO 2: 생성형 AI가 출력한 답변 및 원문 대조(Fact-check) 검증 내역을 기록하세요.
mission_02_result = """
# 여기에 AI 생성 결과 및 팩트체크 대조표/산출물을 마크다운 서식으로 기록하세요.
"""

# 렌더러 실행 (작성한 내용을 시각적으로 확인합니다)
show_result("미션 02. 행정 절차 FAQ 정리하기", mission_02_prompt, mission_02_result)

